In [1]:
# Get the necessary packages

!pip install esm
!pip install peft
!pip install bitsandbytes

from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig
import torch
import torch.nn as nn
from esm.tokenization import EsmSequenceTokenizer
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from peft import LoraConfig, get_peft_model
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import bitsandbytes as bnb
from tqdm import tqdm
from scipy import stats



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 85.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 108.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.7/302.7 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 87.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 143.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 81.

In [2]:
# Set random seeds for reproducibility of your trainings run
def set_seeds(s):
    torch.manual_seed(s)
    np.random.seed(s)

set_seeds(42)

In [3]:
# Load the data
data = pd.read_csv("three_vs_rest.csv")
print(data)
data = data.rename(columns={"target": "label"})
# Split the data into train and test sets
train_df = data[data["set"]=="train"]
test_df = data[data["set"]=="test"]

train_df = train_df.drop(columns=["set","validation"])
test_df = test_df.drop(columns=["set","validation"])

# Print the shape of the data
print("Train set size:", train_df.shape[0])
print("Test set size:", test_df.shape[0])
# Print the columns of the data
print("Columns in train set and test set:", train_df.columns.tolist(), test_df.columns.tolist())

                                               sequence    target    set  \
0     MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYD...  1.000000  train   
1     MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGIDGEWTYD...  1.445905  train   
2     MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGLDGEWTYD...  1.690164  train   
3     MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGMDGEWTYD...  1.170550  train   
4     MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVAGEWTYD...  2.401243  train   
...                                                 ...       ...    ...   
8728  MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGYYSEWTYD...  0.368577   test   
8729  MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGYYVEWTYD...  1.044870   test   
8730  MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGYYVEWTYD...  0.002253   test   
8731  MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGYYYEWTYD...  0.026282   test   
8732  MQYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGYYYEWTYD...  0.004493   test   

     validation  
0           NaN  
1           NaN  
2           NaN  
3           NaN

In [4]:
## Tokenize the sequences

# Shuffle train_df
train_df = train_df.sample(frac=1, random_state=42)

# Reset the index of the shuffled DataFrame
train_df = train_df.reset_index(drop=True)

# Instantiate the tokenizer
tokenizer = EsmSequenceTokenizer()

# Pre-tokenize all sequences in the dataframe
train_df["tokenized_sequence"] = train_df["sequence"].apply(lambda seq: tokenizer(seq, return_tensors="pt"))
test_df["tokenized_sequence"] = test_df["sequence"].apply(lambda seq: tokenizer(seq, return_tensors="pt"))

# Define a PyTorch Dataset class for the sequences
class SequenceDataset(Dataset):
    """Custom PyTorch Dataset for protein sequences."""
    def __init__(self, dataframe):
        self.df = dataframe

    def __len__(self):
        return len(self.df)

    def __getitem__(self,idx):
        row = self.df.iloc[idx]
        sequence = row["sequence"]
        label = row["label"]
        # Use  the pre-tokenized sequence
        sequence_tensor = row["tokenized_sequence"]
        label_tensor = torch.tensor(row["label"], dtype=torch.float)
        return sequence_tensor, label_tensor

# Define a custom collate function to handle padding
def collate_fn(batch):
    """
    Collate function to pad sequences within a batch.
    """
    # batch is a list of tuples: [(sequence_tensor_1, label_tensor_1), (sequence_tensor_2, label_tensor_2), ...]
    sequences = [item[0] for item in batch]
    labels = torch.stack([item[1] for item in batch])

    # Pad the sequences to the maximum length in the batch
    # tokenizer returns a dictionary with 'input_ids' and 'attention_mask'
    input_ids = [seq['input_ids'].squeeze(0) for seq in sequences] # Remove the batch dimension from the tokenizer output
    attention_masks = [seq['attention_mask'].squeeze(0) for seq in sequences]

    # Pad input_ids and attention_masks
    padded_input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    padded_attention_masks = torch.nn.utils.rnn.pad_sequence(attention_masks, batch_first=True, padding_value=0) # Padding value for attention mask is 0

    # Recreate the dictionary structure expected by the model
    padded_sequences = {'input_ids': padded_input_ids, 'attention_mask': padded_attention_masks}

    return padded_sequences, labels


In [5]:
def train_step(model, dataloader, loss_fn, optimizer, device, accumulation_steps=1):
    """
    Trains a PyTorch model for a single epoch using regression loss.

    Args:
        model: A PyTorch model to be trained.
        dataloader: A DataLoader instance for the model to be trained on.
        loss_fn: A PyTorch loss function to minimize (e.g., MSELoss).
        optimizer: A PyTorch optimizer to help minimize the loss function.
        device: A target device to compute on (e.g., "cuda" or "cpu").
        accumulation_steps: The number of steps to accumulate gradients over.

    Returns:
        A float representing the average training loss over the epoch.
    """
    # Put model in train mode
    model.train()

    # Set train loss values
    train_loss = 0

    # Loop through data loader
    for batch, (sequence, labels) in enumerate(tqdm(dataloader, desc="Processing batches for training")):
        # Move sequence tensors and labels to the device
        sequence = {key: val.to(device) for key, val in sequence.items()}
        labels = labels.to(device)

        # Forward pass
        predictions = model(sequence)

        # Calculate loss and normalize for accumulation
        # Use .squeeze() to remove a dimension if the output is (batch_size, 1)
        loss = loss_fn(predictions.squeeze(), labels)

        # If performing gradient accumulation, normalize the loss
        if accumulation_steps > 1:
            loss = loss / accumulation_steps

        # Backward pass
        loss.backward()

        # Perform optimizer step and zero gradients only after accumulation_steps
        if (batch + 1) % accumulation_steps == 0 or (batch + 1) == len(dataloader):
            optimizer.step()
            optimizer.zero_grad()

        # Accumulate total loss before normalization
        train_loss += loss.item()

    # Get average metrics per batch
    train_loss = train_loss / len(dataloader)
    return train_loss

def test_step(model, dataloader, loss_fn, device):
    """
    Tests a PyTorch model for a single epoch and calculates regression metrics.

    Args:
        model: A PyTorch model to be tested.
        dataloader: A DataLoader instance for the model to be tested on.
        loss_fn: A PyTorch loss function to calculate loss on the test data.
        device: A target device to compute on (e.g., "cuda" or "cpu").

    Returns:
        A tuple of testing loss, validation loss and Spearman's rank correlation coefficient.
    """

    # Put model in eval mode
    model.eval()

    # Set test loss value
    test_loss = 0
    # Create lists for actual and predicted labels across all batches
    label_batch = []
    pred_batch = []

    with torch.inference_mode():
        # Loop through dataloader batches
        for batch, (sequence, labels) in enumerate(tqdm(dataloader, desc="Processing batches for testing")):
            # Move sequence tensors and labels to the device
            sequence = {key: val.to(device) for key, val in sequence.items()}
            labels = labels.to(device)

            # Forward pass
            predictions = model(sequence)

            # Calculate and accumulate loss
            # Use .squeeze() to remove a dimension if the output is (batch_size, 1)
            loss = loss_fn(predictions.squeeze(), labels)
            test_loss += loss.item()

            # Accumulate labels and predictions for global metric calculation
            label_batch.extend(labels.tolist())
            pred_batch.extend(predictions.squeeze().tolist())

    # Get average loss per batch
    test_loss = test_loss / len(dataloader)

    # Calculate overall regression metrics for all batches combined
    test_mae = mean_absolute_error(label_batch, pred_batch)
    spearmanr = round(stats.spearmanr(label_batch, pred_batch).statistic,6)


    # Return the calculated regression metrics
    return test_loss, spearmanr


In [6]:
def train(model, train_dataloader, test_dataloader, optimizer, loss_fn, epochs, device, accumulation_steps=1):
  """Trains and tests a PyTorch model.

    Args:
    model: A PyTorch model to be trained and tested.
    train_dataloader: A DataLoader instance for the model to be trained on.
    test_dataloader: A DataLoader instance for the model to be tested on.
    optimizer: A PyTorch optimizer to help minimize the loss function.
    loss_fn: A PyTorch loss function to calculate loss on both datasets.
    epochs: An integer indicating how many epochs to train for.
    device: A target device to compute on (e.g. "cuda" or "cpu").
    accumulation_steps: The number of steps to accumulate gradients over.
                            Default is 1 (no accumulation).

    Returns:
    A dictionary of training and testing loss as well as training and
    testing metrics. Each metric has a value in a list for each epoch.
    """

  results ={
      "Epoch": [],
      "Training Loss": [],
      "Validation Loss": [],
      "Spearmanr": []
  }

  # Set model type as float32 and transfer to device
  model.to(torch.float32).to(device)

  # Loop through training and testing steps for a number of epochs
  for epoch in range(epochs):
    train_loss = train_step(model=model,
                            dataloader=train_dataloader,
                            loss_fn=loss_fn,
                            optimizer=optimizer,
                            device=device,
                            accumulation_steps=accumulation_steps)
    test_loss, spearmanr = test_step(model=model,
                                    dataloader=test_dataloader,
                                    loss_fn=loss_fn,
                                    device=device)

    # Update results dictionary
    results["Epoch"].append(epoch + 1)
    results["Training Loss"].append(train_loss)
    results["Validation Loss"].append(test_loss)
    results["Spearmanr"].append(spearmanr)

    # Display results after each epoch
    epoch_results_df = pd.DataFrame({
        "Epoch": [epoch + 1],
        "Training Loss": [train_loss],
        "Validation Loss": [test_loss],
        "Spearmanr": [spearmanr]
    })
    display(epoch_results_df.set_index("Epoch"))

  # Return the results at the end of the epochs
  return results

In [7]:
# Define  modified ESMC model for protein sequence classification and regression tasks

class ESMC_SequenceClassification(nn.Module):
    '''ESMC model for sequence regression.
    Args:
        num_labels (int): Number of output labels for classification.
        model (ESMC): Pretrained ESMC model instance.
        num_features (int): Dimensionality of the model's output features
        dropout (float): Dropout rate for regularization.
    '''
    def __init__(self, num_labels, model, num_features, dropout=0.0):
        super(ESMC_SequenceClassification, self).__init__()
        self.num_labels = num_labels
        self.model = model
        self.dropout = dropout
        self.num_features = num_features
        self.classification_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(num_features, num_features),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(num_features,num_labels)
            )


    # Modify forward to accept input_ids and attention_mask
    def forward(self, encoded_inputs):
        # Pass input_ids and attention_mask to the base ESM model's standard forward.
        attention_mask = None
        try:
            input_ids = encoded_inputs['input_ids']
            attention_mask = encoded_inputs['attention_mask']
        except KeyError:
             # If encoded_inputs is not a dictionary, assume it's just input_ids
             input_ids = encoded_inputs
             attention_mask = None # Set attention_mask to None if not provided

        # Pass input_ids and attention_mask to the underlying model
        outputs = self.model(sequence_tokens=input_ids, sequence_id=attention_mask)
        #seq_output = outputs.embeddings[:, 0, :]

         #Use mean pooling of the embeddings for sequence-level representation
         #outputs.embeddings will have shape (batch_size, sequence_length, hidden_size)
         #Apply attention mask during pooling
        if attention_mask is not None:
            # Mask out padding tokens before averaging
            masked_embeddings = outputs.embeddings * attention_mask.unsqueeze(-1)
            sum_embeddings = masked_embeddings.sum(dim=1)
            num_non_padded_tokens = attention_mask.sum(dim=1).unsqueeze(-1)
            # Avoid division by zero if a sequence is all padding
            num_non_padded_tokens = torch.max(num_non_padded_tokens, torch.ones_like(num_non_padded_tokens))
            pooled_output = sum_embeddings / num_non_padded_tokens
        else:
            pooled_output = outputs.embeddings.mean(dim=1) # Fallback if no attention mask

        logits = self.classification_head(pooled_output)
        return logits

In [8]:
# Function to get percentage of trainable parameters
def print_trainable_parameters(model):
    trainable_params, all_param = 0, 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param:.2f}")

In [ ]:
## Transfer learning for regression task without fine tuning using ESMC 300M model

# Create datasets for training and testing
train_dataset = SequenceDataset(train_df)
test_dataset = SequenceDataset(test_df)


# Create DataLoaders for training and testing

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn, pin_memory=True,num_workers=2,persistent_workers=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn, pin_memory=True,num_workers=2, persistent_workers=True)

# Load ESMC 300m model from pretrained
model = ESMC.from_pretrained("esmc_300m")

# Freeze all base layers in the model
for param in model.parameters():
    param.requires_grad = False

# Instantiate the ESMC_SequenceClassification model
model = ESMC_SequenceClassification(num_labels=1, model=model, num_features=960)

# Print trainable parameters
print_trainable_parameters(model)

# Select device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Define loss and optimizer
loss_fn = nn.MSELoss()
optimizer = bnb.optim.PagedAdamW8bit(model.parameters(), lr=3e-4)

# Setup training and save the results
results = train(model=model,
                train_dataloader=train_loader,
                test_dataloader=test_loader,
                optimizer=optimizer,
                loss_fn=loss_fn,
                epochs=20,
                device=device)
results = pd.DataFrame(results)
results.to_csv("ESMC_300M_regression_training_logs.csv")

# Save the model state
PATH = "ESMC_300M_regression_model.pth"
torch.save(model.state_dict(), PATH)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

,Training Loss,Validation Loss,Spearmanr
Epoch,,,
18,0.911817,1.216918,0.665627


Processing batches for testing: 100%|██████████| 718/718 [05:32<00:00,  2.16it/s]


,Training Loss,Validation Loss,Spearmanr
Epoch,,,
19,0.904842,1.204122,0.666546


Processing batches for testing: 100%|██████████| 718/718 [05:33<00:00,  2.16it/s]


,Training Loss,Validation Loss,Spearmanr
Epoch,,,
20,0.897863,1.191221,0.667337


In [ ]:
## Fine tuning for regression task using ESMC 300M model with LoRA

# Clear out GPU cache
torch.cuda.empty_cache()

# Create datasets for training and testing
train_dataset = SequenceDataset(train_df)
test_dataset = SequenceDataset(test_df)


# Create DataLoaders for training and testing

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn, pin_memory=True,num_workers=2,persistent_workers=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn, pin_memory=True,num_workers=2, persistent_workers=True)

# Load the ESMC base model from pretrained using the custom class
base_esmc_model = ESMC.from_pretrained("esmc_300m")

# Define the LoRA configuration
lora_config = LoraConfig(
    r=4,
    lora_alpha=1,
    target_modules=[
        "layernorm_qkv.1",
        "out_proj",
        "ffn.1",
        "ffn.3"],
    bias="all"
)

# Use get_peft_model instead of inject_adapter_in_model
base_esmc_model = get_peft_model(base_esmc_model, lora_config)

# Instantiate the ESMC_SequenceClassification model
model = ESMC_SequenceClassification(num_labels=1, model= base_esmc_model, num_features=960)

# Print the trainable parameters
print_trainable_parameters(model)

# Select device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Define loss and optimizer
loss_fn = nn.MSELoss()
optimizer = bnb.optim.PagedAdamW8bit(model.parameters(), lr=3e-4)

# Setup training and save the results
results = train(model=model,
                train_dataloader=train_loader,
                test_dataloader=test_loader,
                optimizer=optimizer,
                loss_fn=loss_fn,
                epochs=10,
                device=device)
results = pd.DataFrame(results)
results.to_csv("ESMC_300M_regression_training_logs_fine_tuned.csv")

# Save the model state
#PATH = "ESMC_300M_regression_model_fine_tuned.pth"
#torch.save(model.state_dict(), PATH)

## Saving checpoint for further training

# Define path and checkpoint content
PATH = "ESMC_300M_regression_model_fine_tuned_checkpoint.pth"
epoch = results["Epoch"].tail(1)
loss = results["Training Loss"].tail(1)

checkpoint = {
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss,
}

# Save the checkpoint
torch.save(checkpoint, PATH)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

...

,Training Loss,Validation Loss,Spearmanr
Epoch,,,
8,0.138947,0.500531,0.874408


Processing batches for testing: 100%|██████████| 718/718 [06:30<00:00,  1.84it/s]


,Training Loss,Validation Loss,Spearmanr
Epoch,,,
9,0.111939,0.440575,0.879866


Processing batches for testing: 100%|██████████| 718/718 [06:30<00:00,  1.84it/s]


,Training Loss,Validation Loss,Spearmanr
Epoch,,,
10,0.100919,0.487311,0.871804


In [ ]:
## Transfer learning for regression task without fine tuning using ESMC 600M model

# Create datasets for training and testing
train_dataset = SequenceDataset(train_df)
test_dataset = SequenceDataset(test_df)


# Create DataLoaders for training and testing

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn, pin_memory=True,num_workers=2,persistent_workers=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn, pin_memory=True,num_workers=2, persistent_workers=True)

# Load ESMC 300m model from pretrained
model = ESMC.from_pretrained("esmc_600m")

# Freeze all base layers in the model
for param in model.parameters():
    param.requires_grad = False

# Instantiate the ESMC_SequenceClassification model
model = ESMC_SequenceClassification(num_labels=1, model=model, num_features=1152)

# Print trainable parameters
print_trainable_parameters(model)

# Select device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Define loss and optimizer
loss_fn = nn.MSELoss()
optimizer = bnb.optim.PagedAdamW8bit(model.parameters(), lr=3e-4)

# Setup training and save the results
results = train(model=model,
                train_dataloader=train_loader,
                test_dataloader=test_loader,
                optimizer=optimizer,
                loss_fn=loss_fn,
                epochs=10,
                device=device)
results = pd.DataFrame(results)
results.to_csv("ESMC_600M_regression_training_logs.csv")

# Save the model state
#PATH = "ESMC_600M_regression_model.pth"
#torch.save(model.state_dict(), PATH)

## Saving checpoint for further training

# Define path and checkpoint content
PATH = "ESMC_600M_regression_model_checkpoint.pth"
epoch = results["Epoch"].tail(1)
loss = results["Training Loss"].tail(1)

checkpoint = {
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss,
}

# Save the checkpoint
torch.save(checkpoint, PATH)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

...

,Training Loss,Validation Loss,Spearmanr
Epoch,,,
8,0.913808,1.18298,0.637126


Processing batches for testing: 100%|██████████| 718/718 [09:42<00:00,  1.23it/s]


,Training Loss,Validation Loss,Spearmanr
Epoch,,,
9,0.899401,1.158697,0.646585


Processing batches for testing: 100%|██████████| 718/718 [09:43<00:00,  1.23it/s]


,Training Loss,Validation Loss,Spearmanr
Epoch,,,
10,0.887728,1.141293,0.653796


In [ ]:
## Fine tuning for regression task using ESMC 600M model with LoRA

# Clear out GPU cache
torch.cuda.empty_cache()

# Create datasets for training and testing
train_dataset = SequenceDataset(train_df)
test_dataset = SequenceDataset(test_df)


# Create DataLoaders for training and testing

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn, pin_memory=True,num_workers=2,persistent_workers=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn, pin_memory=True,num_workers=2, persistent_workers=True)

# Load the ESMC base model from pretrained using the custom class
base_esmc_model = ESMC.from_pretrained("esmc_600m")

# Define the LoRA configuration
lora_config = LoraConfig(
    r=4,
    lora_alpha=1,
    target_modules=[
        "layernorm_qkv.1",
        "out_proj",
        "ffn.1",
        "ffn.3"],
    bias="all"
)

# Use get_peft_model instead of inject_adapter_in_model
base_esmc_model = get_peft_model(base_esmc_model, lora_config)

# Instantiate the ESMC_SequenceClassification model
model = ESMC_SequenceClassification(num_labels=1, model= base_esmc_model, num_features=1152)

# Print the trainable parameters
print_trainable_parameters(model)

# Select device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Define loss and optimizer
loss_fn = nn.MSELoss()
optimizer = bnb.optim.PagedAdamW8bit(model.parameters(), lr=3e-4)

# Setup training and save the results
results = train(model=model,
                train_dataloader=train_loader,
                test_dataloader=test_loader,
                optimizer=optimizer,
                loss_fn=loss_fn,
                epochs=10,
                device=device)
results = pd.DataFrame(results)
results.to_csv("ESMC_600M_regression_training_logs_fine_tuned.csv")

# Save the model state
#PATH = "ESMC_600M_regression_model_fine_tuned.pth"
#torch.save(model.state_dict(), PATH)

## Saving checpoint for further training

# Define path and checkpoint content
PATH = "ESMC_600M_regression_model_fine_tuned_checkpoint.pth"
epoch = results["Epoch"].tail(1)
loss = results["Training Loss"].tail(1)

checkpoint = {
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': loss,
}

# Save the checkpoint
torch.save(checkpoint, PATH)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

,Training Loss,Validation Loss,Spearmanr
Epoch,,,
8,0.292609,0.518748,0.861094


Processing batches for testing: 100%|██████████| 718/718 [10:46<00:00,  1.11it/s]


,Training Loss,Validation Loss,Spearmanr
Epoch,,,
9,0.249103,0.614609,0.86464


Processing batches for testing: 100%|██████████| 718/718 [10:47<00:00,  1.11it/s]


,Training Loss,Validation Loss,Spearmanr
Epoch,,,
10,0.222075,0.574152,0.87099
